# Notebook 1: NLP Preprocessing Pipeline

**Technique covered:** Natural Language Processing (NLP)

This notebook demonstrates the full NLP preprocessing pipeline used in the AI Interview Avatar Agent.
Each candidate response passes through this pipeline before evaluation.

## Pipeline steps
1. Text cleaning (lowercasing, noise removal)
2. Tokenisation (NLTK `word_tokenize`)
3. Stop-word removal
4. Lemmatisation (WordNetLemmatizer)
5. Keyword extraction (frequency-based)
6. Semantic embeddings (OpenAI `text-embedding-3-large`, 3072-dim)
7. Cosine similarity between candidate and ideal answers

In [1]:
# Install dependencies if running standalone
# !pip install nltk openai numpy python-dotenv pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv('../.env')

import nltk
import re
import string
import numpy as np
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

for r in ['punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(r, quiet=True)

print('NLTK ready')

NLTK ready


## 1. Text Cleaning

In [3]:
raw_answer = "  A stack uses LIFO -- Last In, First Out!! It's used for things like... function call stacks, & undo-redo operations.  "

def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s.,?!'\\-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

clean = clean_text(raw_answer)
print(f'Raw    : {repr(raw_answer)}')
print(f'Cleaned: {repr(clean)}')

Raw    : "  A stack uses LIFO -- Last In, First Out!! It's used for things like... function call stacks, & undo-redo operations.  "
Cleaned: "a stack uses lifo -- last in, first out!! it's used for things like... function call stacks, undo-redo operations."


## 2. Tokenisation

In [4]:
tokens = word_tokenize(clean)
sentences = sent_tokenize(clean)

print(f'Token count : {len(tokens)}')
print(f'Tokens      : {tokens}')
print(f'Sentences   : {sentences}')

Token count : 26
Tokens      : ['a', 'stack', 'uses', 'lifo', '--', 'last', 'in', ',', 'first', 'out', '!', '!', 'it', "'s", 'used', 'for', 'things', 'like', '...', 'function', 'call', 'stacks', ',', 'undo-redo', 'operations', '.']
Sentences   : ['a stack uses lifo -- last in, first out!!', "it's used for things like... function call stacks, undo-redo operations."]


## 3. Stop-Word Removal

In [5]:
stop_words = set(stopwords.words('english'))

filtered = [t for t in tokens if t not in stop_words and t not in string.punctuation]

removed = set(tokens) - set(filtered)
print(f'Stop words removed : {sorted(removed)}')
print(f'Remaining tokens   : {filtered}')

Stop words removed : ['!', ',', '.', 'a', 'for', 'in', 'it', 'out']
Remaining tokens   : ['stack', 'uses', 'lifo', '--', 'last', 'first', "'s", 'used', 'things', 'like', '...', 'function', 'call', 'stacks', 'undo-redo', 'operations']


## 4. Lemmatisation

In [6]:
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(t) for t in filtered]

# Show before/after
changes = [(a, b) for a, b in zip(filtered, lemmatized) if a != b]
print('Lemmatisation changes:', changes)
print('Lemmatized tokens    :', lemmatized)

Lemmatisation changes: [('uses', 'us'), ('things', 'thing'), ('stacks', 'stack'), ('operations', 'operation')]
Lemmatized tokens    : ['stack', 'us', 'lifo', '--', 'last', 'first', "'s", 'used', 'thing', 'like', '...', 'function', 'call', 'stack', 'undo-redo', 'operation']


## 5. Keyword Extraction (Frequency-Based)

In [7]:
freq = Counter(lemmatized)
top_keywords = [w for w, _ in freq.most_common(8)]

print('Top keywords:', top_keywords)
print('Frequencies :', dict(freq.most_common(8)))

Top keywords: ['stack', 'us', 'lifo', '--', 'last', 'first', "'s", 'used']
Frequencies : {'stack': 2, 'us': 1, 'lifo': 1, '--': 1, 'last': 1, 'first': 1, "'s": 1, 'used': 1}


## 6. Semantic Embeddings — OpenAI text-embedding-3-large

In [8]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_embedding(text: str) -> np.ndarray:
    result = client.embeddings.create(model="text-embedding-3-large", input=text)
    return np.array(result.data[0].embedding)

candidate_emb = get_embedding(clean)
print(f'Embedding dimensions : {len(candidate_emb)}')
print(f'First 5 values       : {candidate_emb[:5].round(4)}')

Embedding dimensions : 3072
First 5 values       : [-0.0724  0.0228 -0.0212  0.0107 -0.0124]


## 7. Cosine Similarity — Candidate vs Ideal Answer

In [9]:
ideal_answer = "A stack is a LIFO data structure — last in, first out. It is used for function call stacks, expression evaluation, and undo operations. A queue is FIFO — first in, first out — used for BFS and task scheduling."
poor_answer  = "I am not sure, maybe it is something related to databases or files."

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

ideal_emb = get_embedding(ideal_answer)
poor_emb  = get_embedding(poor_answer)

sim_good = cosine_similarity(candidate_emb, ideal_emb)
sim_poor = cosine_similarity(poor_emb, ideal_emb)

print(f'Candidate similarity to ideal : {sim_good:.4f}')
print(f'Poor answer similarity to ideal: {sim_poor:.4f}')
verdict = 'closer' if sim_good > sim_poor else 'further'
print(f'\nConclusion: Candidate answer is {verdict} to the ideal.')

Candidate similarity to ideal : 0.7772
Poor answer similarity to ideal: 0.1875

Conclusion: Candidate answer is closer to the ideal.


## 8. Full Pipeline Demo — Multiple Answers

In [10]:
import pandas as pd

sample_answers = [
    ("Strong",  "A stack is LIFO — last in first out. Used for call stacks, depth-first search, and undo functionality. Operations are O(1) for push and pop."),
    ("Medium",  "Stack stores data and pops the last element. Queue goes in order. I think queue is used for printing."),
    ("Weak",    "Stack is like a pile of plates. I don't remember the rest."),
]

rows = []
for label, ans in sample_answers:
    emb = get_embedding(clean_text(ans))
    sim = cosine_similarity(emb, ideal_emb)
    proc = [lemmatizer.lemmatize(t) for t in word_tokenize(clean_text(ans)) if t not in stop_words]
    kw   = [w for w, _ in Counter(proc).most_common(5)]
    rows.append({'Label': label, 'Cosine Sim': round(sim, 4), 'Keywords': ', '.join(kw)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

 Label  Cosine Sim                     Keywords
Strong      0.7724      ., stack, ,, lifo, last
Medium      0.6695 ., queue, stack, store, data
  Weak      0.4723  ., stack, like, pile, plate


## Summary

| Step | Technique | Purpose |
|---|---|---|
| Cleaning | Regex normalisation | Remove noise, standardise case |
| Tokenisation | NLTK word_tokenize | Split into processable units |
| Stop-word removal | NLTK stopwords | Keep only meaningful tokens |
| Lemmatisation | WordNetLemmatizer | Normalise morphological variants |
| Keywords | Frequency Counter | Identify salient concepts |
| Embeddings | OpenAI text-embedding-3-large (3072-dim) | Dense semantic representation |
| Similarity | Cosine similarity | Measure semantic closeness to ideal |